# Part A

In [ ]:
gauss = [-84.7359, -84.7602, -84.6243, -84.6633, -84.7073]
mog = [-80.1983, -80.2990, -80.2530, -80.2611, -80.2027]
flow = [-77.2721, -77.2764, -77.3146, -77.1605, -77.3165]

In [4]:
import numpy as np

gauss_mean = np.mean(gauss)
mog_mean = np.mean(mog)
flow_mean = np.mean(flow)

gauss_std = np.std(gauss)
mog_std = np.std(mog)
flow_std = np.std(flow)

print(f"Gaussian Prior: Mean = {gauss_mean:.4f}, Std = {gauss_std:.4f}")
print(f"Mixture of Gaussians Prior: Mean = {mog_mean:.4f}, Std = {mog_std:.4f}")
print(f"Planar Flow Prior: Mean = {flow_mean:.4f}, Std = {flow_std:.4f}")

Gaussian Prior: Mean = -84.6982, Std = 0.0490
Mixture of Gaussians Prior: Mean = -80.2428, Std = 0.0379
Planar Flow Prior: Mean = -77.2680, Std = 0.0569


# Part B

In [5]:
# Cell 1 - Imports
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from fid import compute_fid
from vae_part_B import DDPM, BetaVAE, GaussianDecoder, FcNetwork
from vae_part_A import GaussianPrior, GaussianEncoder
from unet import Unet

device = 'cuda'
M = 10
D = 784
T = 1000
beta_values = [1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3]

In [6]:
# Cell 2 - Real images
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x - 0.5) * 2.0),
])
test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('data/', train=False, download=True, transform=transform),
    batch_size=10000
)
x_real = next(iter(test_loader))[0][:10000].view(-1, 1, 28, 28).to(device)

In [7]:
# Cell 3 - U-Net FID
unet = DDPM(Unet(), T=T).to(device)
unet.load_state_dict(torch.load('models/PartB/model_unet.pt', map_location=device))
unet.eval()

with torch.no_grad():
    x_gen = unet.sample((1000, D)).view(-1, 1, 28, 28)
    fid = compute_fid(x_real, x_gen, device=device)
    print(f"U-Net FID: {fid:.2f}")

U-Net FID: 104.36


In [ ]:
# Cell 4 - Beta-VAE and Latent DDPM FID
def make_beta_vae(beta):
    encoder_net = nn.Sequential(
        nn.Flatten(), nn.Linear(784, 512), nn.ReLU(),
        nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, M*2),
    )
    decoder_net = nn.Sequential(
        nn.Linear(M, 512), nn.ReLU(),
        nn.Linear(512, 512), nn.ReLU(),
        nn.Linear(512, 784), nn.Unflatten(-1, (28, 28))
    )
    return BetaVAE(GaussianEncoder(encoder_net), GaussianDecoder(decoder_net), GaussianPrior(M), beta=beta).to(device)

for beta in beta_values:
    beta_vae = make_beta_vae(beta)
    beta_vae.load_state_dict(torch.load(f'models/PartB/model_beta_{beta}_vae.pt', map_location=device))
    beta_vae.eval()

    ddpm_latent = DDPM(FcNetwork(M, 256), T=T).to(device)
    ddpm_latent.load_state_dict(torch.load(f'models/PartB/model_latent_ddpm_{beta}.pt', map_location=device))
    ddpm_latent.eval()

    with torch.no_grad():
        # Beta-VAE FID
        x_gen_vae = (beta_vae.sample(1000).view(-1, 1, 28, 28).clamp(0, 1) - 0.5) * 2.0
        fid_vae = compute_fid(x_real, x_gen_vae, device=device)

        # Latent DDPM FID
        z = ddpm_latent.sample((1000, M))
        x_gen_latent = (beta_vae.decoder(z).mean.view(-1, 1, 28, 28).clamp(0, 1) - 0.5) * 2.0
        fid_latent = compute_fid(x_real, x_gen_latent, device=device)

    print(f"β={beta} | Beta-VAE FID: {fid_vae:.2f} | Latent DDPM FID: {fid_latent:.2f}")

β=1e-08 | Beta-VAE FID: 289.25 | Latent DDPM FID: 71.69
β=1e-07 | Beta-VAE FID: 311.28 | Latent DDPM FID: 52.45
β=1e-06 | Beta-VAE FID: 298.00 | Latent DDPM FID: 39.75
β=1e-05 | Beta-VAE FID: 285.23 | Latent DDPM FID: 39.46
β=0.0001 | Beta-VAE FID: 285.24 | Latent DDPM FID: 36.73
β=0.001 | Beta-VAE FID: 277.07 | Latent DDPM FID: 38.53
